# TP2 - Partie 8 : Fine-tuning mono-task (PLM)

## Importations

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import numpy as np
import pandas as pd

## Charger le modèle et tokenizer

In [ ]:
model_name = "aubmindlab/bert-base-arabertv02-twitter"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## Fonction de préparation des données

In [ ]:
def prepare_data(df):
    """Préparer les données pour le modèle."""
    texts = df['cleaned_tweet'].tolist()
    labels = df['sarcasm'].tolist()

    encodings = tokenizer(texts, truncation=True, padding=True)
    return {
        'input_ids': torch.tensor(encodings['input_ids']),
        'attention_mask': torch.tensor(encodings['attention_mask']),
        'labels': torch.tensor(labels)
    }

## Entraîner le modèle (simple — 1 epoch)

In [ ]:
train_data = prepare_data(df_train)
test_data = prepare_data(df_test)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    save_steps=500,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

## Entraîner

In [ ]:
trainer.train()

## Sauvegarde des prédictions (après fine-tuning)

In [ ]:
def predict_fine_tuned(model, texts):
    """Prédiction après fine-tuning."""
    inputs = tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()

    predictions = (probs[:, 1] > 0.5).astype(int)
    return predictions

In [ ]:
texts_test = df_test['cleaned_tweet'].tolist()
predictions = predict_fine_tuned(model, texts_test)

## Sauvegarde des prédictions

In [ ]:
df_result = pd.DataFrame({
    'id': df_test['id'],
    'y_pred': predictions
})
df_result.to_csv("predictions/fine_tuning_mono_seed42.csv", index=False)

print("✅ Prédictions sauvegardées dans `predictions/fine_tuning_mono_seed42.csv`.")